# 1. Problema Preditivo

Este projeto tem como objetivo prever a variável `cognitive_performance_score`, que representa o desempenho cognitivo do indivíduo no dia seguinte, com base em informações relacionadas à qualidade do sono, hábitos de vida, indicadores fisiológicos e fatores comportamentais. 

Essa previsão é relevante para apoiar a identificação antecipada de fatores que impactam o desempenho cognitivo, possibilitando a adoção de estratégias para melhorar a qualidade do sono, o bem-estar e a produtividade, além de subsidiar decisões em contextos de saúde, pesquisa e desenvolvimento de soluções voltadas ao monitoramento da saúde do sono.

*OBS: O dataset utilizado neste projeto é sintético e contém 100 mil registros.*

# 2. Diretório Principal

In [ ]:
import sys
from pathlib import Path


# Diretório atual
PROJECT_ROOT = Path.cwd() 

# Sobe um diretório por vez até encontrar a pasta ˜src˜.
while not (PROJECT_ROOT / "src").exists(): 
    PROJECT_ROOT = PROJECT_ROOT.parent  

# Se a pasta principal do projeto ainda não está na lista de locais onde se procura módulos, add
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# 3. Carregando o Dataset

In [ ]:
from src.dataset import carregar_dataset

df = carregar_dataset()
df.head(5)

# 4. Fase 1: Análise Exploratória de Dados (EDA)

## 4.1. Estatística Descritiva

In [ ]:
from src.eda import estatistica_descritiva

estatistica_descritiva(df)#

Os tipos de dados estão coerentes com seus respectivos valores. Portanto, não será necessário tratá-los.

A análise estatística descritiva mostra que o conjunto de dados possui 100.000 observações completas, sem valores ausentes nas variáveis numéricas. A variável-alvo (`cognitive_performance_score`) possui média de 59,23 e mediana de 60,40, sugerindo uma distribuição aproximadamente simétrica.

## 4.2. Visualização de Dados

In [ ]:
from src.eda import visualizar_dados

visualizar_dados(df)

### Histograma de distribuição da variável-alvo

A variável-alvo apresenta distribuição aproximadamente simétrica, com leve assimetria negativa (−0,29), concentrando a maior parte dos valores entre 50 e 80 pontos. Essa característica é favorável para modelos de regressão.

---

### Gráficos de dispersão entre variáveis explicativas e a variável-alvo

**Qualidade do sono × Desempenho cognitivo:** relação positiva, indicando que maiores valores de qualidade do sono tendem a estar associados a um melhor desempenho cognitivo.

**Estresse × Desempenho cognitivo:** relação negativa, sugerindo que maiores níveis de estresse tendem a estar associados a um menor desempenho cognitivo.

---

### Mapa de calor da correlação de Pearson

O mapa de calor evidencia que a variável-alvo apresenta correlação positiva principalmente com a qualidade (muito forte -> 0,85) e a duração do sono, e correlação negativa com o estresse e as horas de trabalho. Além disso, as variáveis explicativas apresentam, em geral, correlações de baixa a moderada intensidade, não sendo observados indícios de multicolinearidade significativa.

## 4.3. Remoção de Coluna

(Removida a coluna **person_id** antes do tratamento e limpeza por motivos de antecipação ao cálculo do VIF mais abaixo)

Esta coluna não contribui no resultado final.

In [ ]:
from src.preprocessing import remover_colunas

df = remover_colunas(df, ["person_id"])

df.head(5)

## 4.4. Reforçando Verificação de Existência de Multicolinearidade

In [ ]:
from src.eda import calcular_vif

vif = calcular_vif(df, variavel_alvo="cognitive_performance_score")
display(vif)

> "Regra prática: VIF acima de **5** já pede atenção; acima de **10** indica redundância séria."

Como não há nenhum VIF acima de 5, não há variável que necessite de atenção em relação a multicolinearidade.

# 5. Fase 2: Tratamento e Limpeza (Data Prep)

## 5.1. Linhas Duplicadas

In [ ]:
from src.preprocessing import verificar_duplicados

verificar_duplicados(df)

Não há registros duplicados. Logo, não há necessidade de tratá-los.

## 5.2. Valores Ausentes (Missing Data)

In [ ]:
from src.preprocessing import verificar_valores_ausentes

verificar_valores_ausentes(df)

Não há valores ausentes. Logo, não há necessidade de tratá-los.

## 5.3. Gerenciamento de Outliers

In [ ]:
from src.preprocessing import plotar_boxplots

plotar_boxplots(df, variavel_alvo="cognitive_performance_score")

Os boxplots indicaram valores extremos em algumas variáveis explicativas. Como a análise não indicou erros de medição nem valores impossíveis, optou-se por mantê-los para preservar a representatividade da base de dados. Embora a Regressão Linear seja sensível a valores extremos, as métricas semelhantes entre treino e teste e a distribuição dos resíduos indicam que esses valores não comprometeram significativamente a capacidade de generalização do modelo.

# 6. Fase 3: Feature Engineering (Coluna Calculada)

Criação da variável `recovery_score`, calculada pela diferença entre a qualidade do sono (`sleep_quality_score`) e o nível de estresse (`stress_score`). Lembrando que ambas as variáveis possuem range de 0 a 10.

- (+) → qualidade do sono maior que o estresse;

-   0 → qualidade do sono e estresse equivalentes;

- (-) → estresse maior que a qualidade do sono.


O objetivo é representar o equilíbrio entre recuperação física/mental e o fator de desgaste estresse.

In [ ]:
df["recovery_score"] = (df["sleep_quality_score"] - df["stress_score"]) # Cria nova coluna

In [ ]:
df.head(5)

In [ ]:
# Visualização com a nova coluna
df[["sleep_quality_score", "stress_score", "recovery_score", "cognitive_performance_score"]].head(10)

In [ ]:
vif = calcular_vif(df, variavel_alvo="cognitive_performance_score")
display(vif)

Nota-se acima que a nova coluna `recovery_score` **gerou multicolinearidade perfeita**. 

In [ ]:
df = remover_colunas(df, ["sleep_quality_score", "stress_score"])

Como a nova variável  `recovery_score` é calculada diretamente a partir dessas duas variáveis, `sleep_quality_score` e `stress_score`, elas foram removidas do conjunto de preditores para remover a multicolinearidade perfeita criada e tornar o modelo mais estável.

In [ ]:
from src.dataset import salvar_dataset_processado

salvar_dataset_processado(df, versao="v1")

Visualiza multicolinearidade novamente:

In [ ]:
vif = calcular_vif(df, variavel_alvo="cognitive_performance_score")
display(vif)

# 7. Fase 4: Preparação para Modelagem

Salva df final que será usado para modelagem.

In [ ]:
from src.dataset import salvar_dataset_final

salvar_dataset_final(df, versao="v1")

Divide conjunto de dados em treino (80%) e teste (20%).

In [ ]:
from src.modeling import dividir_treino_teste

X_train, X_test, y_train, y_test = dividir_treino_teste(df, variavel_alvo="cognitive_performance_score", test_size=0.2, random_state=42)

# 8. Fase 5: Modelagem, Validação e Diagnóstico de Overfitting

## 8.1. Treinamento e Testes

As variáveis categóricas serão tratadas com One-Hot Encoding, utilizando `drop="first"` para evitar redundância entre as categorias e `handle_unknown="ignore"` para permitir categorias não observadas no treino.

As variáveis numéricas serão padronizadas com StandardScaler. O pré-processamento foi inserido em uma Pipeline, garantindo que os parâmetros do escalonamento sejam ajustados apenas com o conjunto de treino e posteriormente aplicados ao conjunto de teste, evitando vazamento de dados.

In [ ]:
from src.modeling import treinar_regressao_linear, realizar_predicao

modelo_rl = treinar_regressao_linear(X_train, y_train)
y_pred    = realizar_predicao(modelo_rl, X_test)

Apenas visualizando algumas previsões (real/previsto):

In [ ]:
import pandas as pd

resultado = pd.DataFrame({"Valor Real": y_test.values, "Valor Previsto": y_pred})

resultado.head(10)

## 8.2. Métricas Técnicas: Comparação Treino e Teste

Comparando métricas nos conjuntos de treino e teste:

In [ ]:
from src.modeling import comparar_metricas_treino_teste

comparacao_metricas = comparar_metricas_treino_teste(modelo_rl, X_train, y_train, X_test, y_test)

display(comparacao_metricas)

> As métricas obtidas nos conjuntos de treino e teste foram muito semelhantes, indicando que o modelo apresentou boa capacidade de generalização. As diferenças observadas foram pequenas (MAE: 0,047; RMSE: 0,076; R²: -0,003), não havendo evidências de overfitting. Isso demonstra que o modelo mantém desempenho consistente ao realizar previsões em dados não utilizados durante o treinamento.

# 9. Fase 6: Avaliação, Interpretação e Versionamento do Modelo

## 9.1. Métricas Técnicas para Conjunto de Teste

In [ ]:
from src.modeling import avaliar_modelo

metricas_teste = avaliar_modelo(y_test, y_pred)

print("=" * 60)
print("MÉTRICAS DA REGRESSÃO LINEAR - CONJUNTO TESTE")
print("=" * 60)

for nome, valor in metricas_teste.items():
    print(f"{nome}: {valor:.4f}")

O modelo erra, em média, cerca de 6 pontos na previsão do desempenho cognitivo.

O R² foi de 0,8807, o que significa que o modelo explica cerca de 88,07% da variabilidade da variável-alvo.

Como o RMSE é apenas um pouco maior que o MAE, isso indica que não existem muitos erros extremamente grandes nas previsões, isto é, os erros do modelo são relativamente uniformes. Se o RMSE fosse muito maior que o MAE, seria um indício de que alguns casos apresentam erros bastante elevados.

## 9.2. Visualização de Valores Reais/Previstos e Resíduos

In [ ]:
from src.modeling import plotar_valores_reais_previstos, plotar_residuos

plotar_valores_reais_previstos(y_test, y_pred)
plotar_residuos(y_test, y_pred)

**Valores reais  x Previstos**: Os pontos estão próximos da linha de referência (linha vermelha), indicando que, na maioria dos casos, as previsões ficaram próximas dos valores reais.

**Resíduos**: Os resíduos estão concentrados próximos de zero, no qual tem uma distribuição próxima de uma normal (sino).

Portanto, há um bom desempenho no modelo.

## 9.3. Veredito de Negócios

> O modelo apresentou um MAE de 6, ou seja, um erro médio de aproximadamente 6 pontos representa cerca de 6% da escala (0–100). Além disso, o R² de 0,8807 indica que o modelo explica aproximadamente 88% da variação dos dados. Dessa forma, o modelo apresenta um bom desempenho e pode ser utilizado como **apoio na análise** da relação entre sono, estilo de vida e desempenho cognitivo, mas **não deve substituir uma avaliação clínica**.

## 9.4. Versionamento do Modelo

Agora, treina o modelo com a base de dados completa e salva modelo e métricas de teste.

In [ ]:
from src.modeling import treinar_regressao_linear, salvar_modelo

X = df.drop(columns=["cognitive_performance_score"])  # Variáveis explicativas
y = df["cognitive_performance_score"]                 # Variável alvo

# Retreina na base completa
modelo_final = treinar_regressao_linear(X, y)

 # As métricas salvas são as do processo de validação do modelo, não as da base completa.
salvar_modelo(
    modelo=modelo_final,
    metricas=metricas_teste,
    variaveis_explicativas=X.columns.tolist(),
) 